## class label for majority of CF users. 0 - hate speech 1 - offensive language 2 - neither


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.pipeline import Pipeline

#to data preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

#NLP tools
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

#train split and fit models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from nltk.tokenize import TweetTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot as plt

import nltk
from nltk.util import pr
from nltk.tokenize import word_tokenize      #to divide strings into lists of substrin
from nltk.corpus import stopwords            #to filterout useless data

import tensorflow as tf
from tensorflow.keras.preprocessing import text, sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, MaxPooling1D

import seaborn as sns
import re
import string

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
twitts_data = pd.read_csv('labeled_data.csv')
twitts_data.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [3]:
class_data = twitts_data['class'].value_counts()
class_data

class
1    19190
2     4163
0     1430
Name: count, dtype: int64

# Start Here

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import wordcloud
from textblob import TextBlob
from nltk.sentiment.util import *

nltk.download('stopwords')
nltk.download('wordnet')
lm = WordNetLemmatizer()
def clean(text):
    text = str(text).lower()
    text = re.sub('[^a-z]', ' ', str(text))
    text = text.split()
    text = [lm.lemmatize(j) for j in text if j not in set(stopwords.words('english'))]
    text = ' '.join(str(k) for k in text)
    return text
df = pd.read_csv('labeled_data.csv')
df = df.drop_duplicates("tweet")
df['cleantext'] = df['tweet'].apply(clean)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [5]:
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet,cleantext
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...,rt mayasolovely woman complain cleaning house ...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,rt mleew boy dat cold tyga dwn bad cuffin dat ...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,rt urkindofbrand dawg rt sbaby life ever fuck ...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,rt c g anderson viva based look like tranny
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,rt shenikaroberts shit hear might true might f...


In [6]:
trump_df = pd.read_csv('trump_US_cleaned.csv')
trump_df.head()

,user_id,likes,retweet_count,cleantext
0,19329393.0,206.0,12.0,without doubt fourseasons debacle best part do...
1,15754281.0,56.0,16.0,president donald trump youngest daughter tiffa...
2,26642006.0,695.0,291.0,woah read article president debt donald trump ...
3,17446621.0,149.0,15.0,latest episode snl tackled donaldtrump loss co...
4,17446621.0,83.0,10.0,nataliemorales explains latinx voter supported...


# New Sets

In [7]:
def check_and_convert(value):
    if isinstance(value, float):  # Check if the value is a float
        return str(value)         # Convert float to string
    else:
        return value

# Apply the function to the DataFrame column
trump_df['cleantext'] = trump_df['cleantext'].apply(lambda x: check_and_convert(x))

In [8]:
X_train = df['cleantext']
Y_train = df['class']

X_test = trump_df['cleantext']

x = X_train.values
y = Y_train.values

x_tokenizer = text.Tokenizer(20000)
x_tokenizer.fit_on_texts(list(x))

x_tokenized = x_tokenizer.texts_to_sequences(x)
x_train_val= sequence.pad_sequences(x_tokenized, maxlen=512)

x_test_tokenized = x_tokenizer.texts_to_sequences(X_test.values)
x_test = sequence.pad_sequences(x_test_tokenized,maxlen=512)

In [9]:
Y_train.values

array([2, 1, 1, ..., 1, 1, 2])

In [11]:
embedding_dim =100
embeddings_index = dict()
f = open('glove.6B.100d.txt')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:],dtype='float32')
    embeddings_index[word]= coefs
f.close()
print(f'Glove model words count {len(embeddings_index)} word vectors')

Glove model words count 216212 word vectors


In [12]:
embedding_matrix = np.zeros((20000, embedding_dim))

for word, index in x_tokenizer.word_index.items():
    if index > 20000 - 1:
        break
    else:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None and len(embedding_vector) == embedding_dim:
            embedding_matrix[index] = embedding_vector


In [13]:
y_train_plus_val =  tf.keras.utils.to_categorical(y, num_classes=3)
#y_test =  tf.keras.utils.to_categorical(Y_test.values, num_classes=3)
y_train_plus_val

array([[0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

In [14]:

#%%
def build_model(embedding_matrix, embedding_dim=100):
    model = Sequential()
    model.add(Embedding(20000,
                        embedding_dim,
                        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                        trainable=False))
    model.add(Dropout(0.2))

    model.add(Conv1D(64, 2, padding='valid', activation='relu'))
    model.add(MaxPooling1D())
    model.add(Conv1D(64, 2, padding='valid', activation='relu'))
    model.add(MaxPooling1D())

    model.add(Conv1D(32, 2, padding='valid', activation='relu'))
    model.add(MaxPooling1D())
    model.add(Conv1D(32, 2, padding='valid', activation='relu'))
    model.add(GlobalMaxPooling1D())

    model.add(Dense(16, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(3, activation='softmax'))

    model.summary()
    return model
model = build_model(embedding_matrix)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 100)         2000000   
                                                                 
 dropout (Dropout)           (None, None, 100)         0         
                                                                 
 conv1d (Conv1D)             (None, None, 64)          12864     
                                                                 
 max_pooling1d (MaxPooling1  (None, None, 64)          0         
 D)                                                              
                                                                 
 conv1d_1 (Conv1D)           (None, None, 64)          8256      
                                                                 
 max_pooling1d_1 (MaxPoolin  (None, None, 64)          0         
 g1D)                                                   

In [15]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [16]:
history = model.fit(x_train_val, y_train_plus_val, batch_size= 64, validation_split=0.2, epochs=10)

Epoch 1/10
310/310 [==============================] - 10s 11ms/step - loss: 0.6117 - accuracy: 0.7808 - val_loss: 0.4089 - val_accuracy: 0.8517
Epoch 2/10
310/310 [==============================] - 3s 10ms/step - loss: 0.4720 - accuracy: 0.8338 - val_loss: 0.3781 - val_accuracy: 0.8687
Epoch 3/10
310/310 [==============================] - 3s 8ms/step - loss: 0.4395 - accuracy: 0.8439 - val_loss: 0.3643 - val_accuracy: 0.8759
Epoch 4/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4175 - accuracy: 0.8501 - val_loss: 0.3526 - val_accuracy: 0.8735
Epoch 5/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4024 - accuracy: 0.8548 - val_loss: 0.3489 - val_accuracy: 0.8771
Epoch 6/10
310/310 [==============================] - 2s 8ms/step - loss: 0.3869 - accuracy: 0.8588 - val_loss: 0.3402 - val_accuracy: 0.8788
Epoch 7/10
310/310 [==============================] - 3s 9ms/step - loss: 0.3795 - accuracy: 0.8607 - val_loss: 0.3378 - val_accuracy: 0.8798
Epo

In [ ]:
#model.evaluate(x_test,y_test, batch_size = 64)

39/39 [==============================] - 0s 10ms/step - loss: 0.6235 - accuracy: 0.7862


[0.6235461831092834, 0.7862040996551514]

In [17]:
x_test

array([[    0,     0,     0, ...,   532,     4,     5],
       [    0,     0,     0, ...,  2180,     4,     5],
       [    0,     0,     0, ...,     5,   289,  7680],
       ...,
       [    0,     0,     0, ...,     4,     5,   231],
       [    0,     0,     0, ...,     4,     5, 12644],
       [    0,     0,     0, ...,  7680,  7680,  7680]], dtype=int32)

In [18]:
y_pred = model.predict(x_test)
y_pred

6561/6561 [==============================] - 17s 2ms/step


array([[0.01143557, 0.19119853, 0.7973659 ],
       [0.01593758, 0.18796347, 0.796099  ],
       [0.02436093, 0.47508255, 0.50055647],
       ...,
       [0.04089392, 0.45723385, 0.50187224],
       [0.01996513, 0.27350044, 0.70653445],
       [0.07727675, 0.73508066, 0.18764266]], dtype=float32)

In [19]:
y_pred = np.array( [ np.argmax (y) for y in y_pred ] )

In [20]:
y_pred

array([2, 2, 2, ..., 2, 2, 1])

In [21]:
trump_df['class'] = y_pred
trump_df

,user_id,likes,retweet_count,cleantext,class
0,1.932939e+07,206.0,12.0,without doubt fourseasons debacle best part do...,2
1,1.575428e+07,56.0,16.0,president donald trump youngest daughter tiffa...,2
2,2.664201e+07,695.0,291.0,woah read article president debt donald trump ...,2
3,1.744662e+07,149.0,15.0,latest episode snl tackled donaldtrump loss co...,2
4,1.744662e+07,83.0,10.0,nataliemorales explains latinx voter supported...,2
...,...,...,...,...,...
209923,9.744188e+17,2.0,0.0,donaldtrump killing america votehimout,1
209924,1.319730e+18,0.0,0.0,watching poor guy foxnews explain reality trum...,1
209925,1.312579e+18,0.0,0.0,trump losing reelection trump biden presidentt...,2
209926,1.313286e+18,0.0,0.0,kayleigh mcenany trump brings big gun kayleigh...,2


In [22]:
trump_df.to_csv('trump_df_civility_lables.csv', index=False)